# MuseTalk Lip-Sync on Colab **T4** (high quality)

MuseTalk replaces Wav2Lip: it works at a **256x256** face region with latent-
space inpainting, so the mouth is far sharper and more natural than Wav2Lip's
blurry 96x96. Give it a video (or image) of a face + an audio clip and it
re-syncs the mouth, keeping the rest of the frame.

### Runs on the Colab T4 only — NOT HuggingFace ZeroGPU.
This clones MuseTalk and runs it **in your Colab runtime** on the T4 Colab
assigns you. It never calls a hosted HF Space, so ZeroGPU is never involved.
Step 0 prints the GPU so you can confirm it's a Tesla T4.

## How to use
1. **Runtime -> Change runtime type -> T4 GPU -> Save.**
2. Run **Step 0** (GPU proof), then **Step 1** (install + launch). Step 1 takes
   ~3-5 min to install and download the models, then prints a **Gradio link**
   (a `https://....gradio.live` URL). Open it.
3. In the MuseTalk UI: upload your **video** (the wide `joker-walk-14b` clip is
   fine — the whole shot is kept) and your **audio** (e.g. floyddeath.wav),
   then Generate. Download the result and move it to `D:\\MatrixVideos`.

> Tip: MuseTalk trained on 25fps video; your Wan clips are already fine. For the
> sharpest talking face, the character being closer to camera always helps, but
> MuseTalk handles small faces much better than Wav2Lip did.


## Step 0 - Prove the GPU is a T4 (not ZeroGPU, not CPU)


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU, then rerun.'
print('Torch is using:', torch.cuda.get_device_name(0), '- Colab hardware, not HF ZeroGPU.')


## Step 1 - Install MuseTalk + download models + launch the UI
One cell (from camenduru's maintained MuseTalk-jupyter — it uses a prebuilt
mmcv wheel to avoid the usual dependency hell). When it finishes it prints a
Gradio link; open that to upload your video + audio. This cell keeps running
while the app is up — that's normal; use the Gradio link, not this cell.


In [ ]:
%cd /content
!git clone -b dev https://github.com/camenduru/MuseTalk
%cd /content/MuseTalk

!pip install -q diffusers==0.27.2 accelerate==0.28.0 omegaconf ffmpeg-python mmpose mmdet gradio
!pip install -q https://github.com/camenduru/wheels/releases/download/colab2/mmcv-2.1.0-cp310-cp310-linux_x86_64.whl

!apt -y install -qq aria2
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MuseTalk/resolve/main/dwpose/dw-ll_ucoco.pth -d /content/MuseTalk/models/dwpose -o dw-ll_ucoco.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MuseTalk/resolve/main/dwpose/dw-ll_ucoco_384.onnx -d /content/MuseTalk/models/dwpose -o dw-ll_ucoco_384.onnx
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MuseTalk/resolve/main/dwpose/dw-ll_ucoco_384.pth -d /content/MuseTalk/models/dwpose -o dw-ll_ucoco_384.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MuseTalk/resolve/main/dwpose/dw-mm_ucoco.pth -d /content/MuseTalk/models/dwpose -o dw-mm_ucoco.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MuseTalk/resolve/main/dwpose/dw-ss_ucoco.pth -d /content/MuseTalk/models/dwpose -o dw-ss_ucoco.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MuseTalk/resolve/main/dwpose/dw-tt_ucoco.pth -d /content/MuseTalk/models/dwpose -o dw-tt_ucoco.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MuseTalk/resolve/main/dwpose/rtm-l_ucoco_256-95bb32f5_20230822.pth -d /content/MuseTalk/models/dwpose -o rtm-l_ucoco_256-95bb32f5_20230822.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MuseTalk/resolve/main/dwpose/rtm-x_ucoco_256-05f5bcb7_20230822.pth -d /content/MuseTalk/models/dwpose -o rtm-x_ucoco_256-05f5bcb7_20230822.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MuseTalk/resolve/main/dwpose/rtm-x_ucoco_384-f5b50679_20230822.pth -d /content/MuseTalk/models/dwpose -o rtm-x_ucoco_384-f5b50679_20230822.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MuseTalk/resolve/main/dwpose/yolox_l.onnx -d /content/MuseTalk/models/dwpose -o yolox_l.onnx
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MuseTalk/resolve/main/face-parse-bisent/79999_iter.pth -d /content/MuseTalk/models/face-parse-bisent -o 79999_iter.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MuseTalk/resolve/main/face-parse-bisent/resnet18-5c106cde.pth -d /content/MuseTalk/models/face-parse-bisent -o resnet18-5c106cde.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MuseTalk/raw/main/musetalk/musetalk.json -d /content/MuseTalk/models/musetalk -o musetalk.json
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MuseTalk/resolve/main/musetalk/pytorch_model.bin -d /content/MuseTalk/models/musetalk -o pytorch_model.bin
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MuseTalk/raw/main/sd-vae-ft-mse/config.json -d /content/MuseTalk/models/sd-vae-ft-mse -o config.json
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MuseTalk/resolve/main/sd-vae-ft-mse/diffusion_pytorch_model.bin -d /content/MuseTalk/models/sd-vae-ft-mse -o diffusion_pytorch_model.bin
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MuseTalk/resolve/main/sd-vae-ft-mse/diffusion_pytorch_model.safetensors -d /content/MuseTalk/models/sd-vae-ft-mse -o diffusion_pytorch_model.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/MuseTalk/resolve/main/whisper/tiny.pt -d /content/MuseTalk/models/whisper -o tiny.pt

!python app.py